# 🚀 Step 99 — Master Orchestrator (Real End-to-End Run)
**Forest Carbon Continuity Pipeline · Sentinel-1/2 Hybrid for EU LULUCF Compliance**

## What this step does
Runs the **entire pipeline for real** — no `time.sleep` simulations: it chains CRS reprojection →
SAR cloud-proof masking → NFI validation → Random Forest species classification → Tier-2 carbon
accounting, then embeds **provenance metadata** (git SHA + input SHA-256) into the final GeoTIFFs.
A deliberately corrupted tile demonstrates the **fail-loud quarantine** protocol. Everything runs
on synthetic data generated in-notebook, so it works on Colab day one with zero setup.

## The Science
An orchestrator is where "handover quality" is won or lost. Three safeguards are exercised here:
- **Fail-loud quarantine** — bad data never fails silently; it is logged to structured JSON
  (`quarantine/fail_*.json` in production) and the run continues.
- **Deterministic replay** — every output GeoTIFF carries `PROVENANCE_COMMIT_SHA` and
  `INPUT_HASH`, so any number in a report can be traced to the exact code + input that produced it.
- **Seeded reproducibility** — one seed regenerates the whole synthetic world, so re-running this
  notebook reproduces every chart and total bit-for-bit.

In [ ]:
# ============================================================
# Environment detection & package setup (runs first, auto-detects
# Google Colab vs local Jupyter - nothing needs manual pip install)
# ============================================================
import sys, subprocess

def _ensure(pkg, import_name=None):
    import_name = import_name or pkg
    try:
        __import__(import_name)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
    colab_files = None

_ensure("numpy"); _ensure("pandas"); _ensure("matplotlib")
_ensure("rasterio"); _ensure("geopandas"); _ensure("scikit-learn")
_ensure("openpyxl"); _ensure("ipywidgets")

def outpath(name):
    """Returns a working save path for the current environment."""
    return f"/content/{name}" if IN_COLAB else name

print(f"Environment detected: {'Google Colab' if IN_COLAB else 'Local Jupyter'}")

# ============================================================
# Unified cross-environment helpers:
# download buttons, regenerate buttons, upload widgets
# ============================================================
import zipfile
import ipywidgets as widgets
from IPython.display import display, FileLink, clear_output

_generated_files = {}  # friendly label -> filepath, used by the final "Download All" bundle

def register_file(label, path):
    if label:
        _generated_files[label] = path

def make_download_button(path, label=None, register_as=None):
    """Cross-environment download button. Colab: real browser download.
    Local Jupyter: clickable FileLink. Reads the file fresh at click time."""
    label = label or f"Download {path}"
    if register_as:
        register_file(register_as, path)
    btn = widgets.Button(description=label, button_style="info", icon="download")
    out = widgets.Output()

    def on_click():
        with out:
            clear_output(wait=True)
            if IN_COLAB:
                colab_files.download(path)
            else:
                display(FileLink(path))
                print(f"Click the link above to download '{path}'.")

    btn.on_click(on_click)
    display(widgets.VBox([btn, out]))

def make_regenerate_button(rerun_fn, label="🔄 Regenerate & Re-run"):
    """Button that calls rerun_fn() (fresh random seed, full re-run of this
    section) and shows the refreshed output inline below the button."""
    btn = widgets.Button(description=label, button_style="warning", icon="refresh")
    out = widgets.Output()

    def on_click():
        with out:
            clear_output(wait=True)
            rerun_fn()

    btn.on_click(on_click)
    display(widgets.VBox([btn, out]))

def make_upload_widget(on_loaded, accept=".csv,.xlsx", label="Upload replacement file"):
    """Colab: native blocking file picker via google.colab.files.upload().
    Local Jupyter: ipywidgets.FileUpload + Load button (non-blocking)."""
    out = widgets.Output()

    if IN_COLAB:
        btn = widgets.Button(description=label, button_style="primary", icon="upload")

        def on_click():
            with out:
                clear_output(wait=True)
                print("Choose a file in the dialog, or Cancel to keep current data.")
                uploaded = colab_files.upload()
                if uploaded:
                    fname = next(iter(uploaded))
                    on_loaded(fname, uploaded[fname])
                else:
                    print("No file selected - keeping current data.")

        btn.on_click(on_click)
        display(widgets.VBox([btn, out]))
    else:
        uploader = widgets.FileUpload(accept=accept, multiple=False, description="Choose file")
        load_btn = widgets.Button(description=label, button_style="primary", icon="upload")

        def on_click():
            with out:
                clear_output(wait=True)
                val = uploader.value
                if not val:
                    print("No file selected yet - choose a file above, then click the button.")
                    return
                item = list(val.values())[0] if isinstance(val, dict) else val[0]
                if isinstance(item, dict) and "metadata" in item:
                    fname = item["metadata"]["name"]     # ipywidgets 7.x shape
                else:
                    fname = item["name"]                  # ipywidgets 8.x shape
                on_loaded(fname, bytes(item["content"]))

        load_btn.on_click(on_click)
        display(widgets.VBox([uploader, load_btn, out]))

def make_download_all_button(label="⬇️ Download ALL outputs (.zip)"):
    """Zips every output registered so far and offers it as one download."""
    zip_path = outpath("outputs_bundle.zip")
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for _label, _path in _generated_files.items():
            try:
                zf.write(_path)
            except FileNotFoundError:
                pass
    make_download_button(zip_path, label=label)

print("Helpers ready: make_download_button / make_regenerate_button / make_upload_widget")

In [ ]:
# ============================================================
# Standalone synthetic inputs for the whole pipeline (Day-1 ready)
# ============================================================
import numpy as np
import pandas as pd
import rasterio
import geopandas as gpd
import matplotlib.pyplot as plt
from rasterio.transform import from_bounds
from rasterio.warp import calculate_default_transform, reproject, Resampling
from sklearn.ensemble import RandomForestClassifier
import hashlib, json, subprocess, time
from datetime import datetime

H, W = 120, 120
BOUNDS = (-6.4, 52.95, -6.1, 53.15)
TRANSFORM = from_bounds(*BOUNDS, W, H)
PIXEL_AREA_HA = 100.0 / 10000.0
BIOMASS_PEAK = {1: 450.0, 2: 380.0, 3: 300.0}
UNCERTAINTY_FACTOR = {1: 0.15, 2: 0.18, 3: 0.25}
SPECIES_NAMES = {1: "Sitka Spruce", 2: "Lodgepole Pine", 3: "Native Mixed"}

def generate_all_inputs(seed=42):
    """One seeded call creates every synthetic input the pipeline needs."""
    rng = np.random.default_rng(seed)
    yy, xx = np.mgrid[0:H, 0:W]

    def blob(cx, cy, sx, sy):
        return np.exp(-(((xx - cx) / sx) ** 2 + ((yy - cy) / sy) ** 2))

    b1, b2, b3 = blob(35, 40, 20, 16), blob(85, 85, 24, 20), blob(80, 25, 18, 18)

    vh = rng.uniform(0.01, 0.15, (H, W)).astype("float32")
    vv = rng.uniform(0.05, 0.25, (H, W)).astype("float32")
    vh[45:75, 45:75] = 0.001; vv[45:75, 45:75] = 0.30   # peat/water block
    nir = np.clip(0.55 + rng.normal(0, 0.08, (H, W)), 0.02, 0.95).astype("float32")
    red = np.clip(0.18 + rng.normal(0, 0.05, (H, W)), 0.01, 0.60).astype("float32")

    n_plots = 60
    nfi = pd.DataFrame({
        "x": rng.uniform(-6.40, -6.10, n_plots).round(6),
        "y": rng.uniform(52.95, 53.15, n_plots).round(6),
        "DBH": rng.uniform(4, 45, n_plots).round(1),
        "Species": rng.choice([1, 2, 3], n_plots, p=[0.55, 0.25, 0.20]),
        "Distance": rng.uniform(5, 25, n_plots).round(1),
        "Angle": rng.uniform(15, 55, n_plots).round(1),
        "ObsHeight": np.full(n_plots, 1.5),
    })

    features = np.stack([
        -13 + 3.0 * b1 + rng.normal(0, 1.0, (H, W)),
        0.20 + 0.10 * b2 + rng.normal(0, 0.02, (H, W)),
        0.75 + 0.10 * b1 - 0.05 * b2 + rng.normal(0, 0.05, (H, W)),
        0.20 + rng.normal(0, 0.03, (H, W)),
        0.60 + rng.normal(0, 0.06, (H, W)),
        rng.normal(0.30, 0.05, (H, W)),
    ]).astype("float32")
    species_truth = (np.argmax(np.stack([b1, b2, b3]), axis=0) + 1).astype("uint8")

    return {"vh": vh, "vv": vv, "nir": nir, "red": red, "nfi": nfi,
            "features": features, "species_truth": species_truth}

inputs = generate_all_inputs(42)
print("Synthetic inputs ready: S1 (VH/VV) + S2 (NIR/Red) + NFI table + feature stack")

In [ ]:
# ============================================================
# Pipeline phases - real processing, same logic as src/*.py
# ============================================================
QUARANTINE = []

def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(65536), b""):
            h.update(chunk)
    return h.hexdigest()

def log_quarantine(tile_id, error_code, details, input_hash="-"):
    """Fail-loud quarantine (mirrors src/quarantine.py): bad tiles are logged, never silent."""
    report = {"tile_id": tile_id, "error_code": error_code, "input_hash": input_hash,
              "timestamp": datetime.now().isoformat(timespec="seconds"), "details": details}
    QUARANTINE.append(report)
    qpath = outpath("quarantine_log.json")
    with open(qpath, "w") as f:
        json.dump(QUARANTINE, f, indent=2)
    print(f"⚠️  QUARANTINE: tile '{tile_id}' failed [{error_code}] - logged, pipeline continues.")
    return qpath

def write_tif(path, array):
    with rasterio.open(path, "w", driver="GTiff", height=array.shape[0], width=array.shape[1],
                       count=1, dtype=str(array.dtype), crs="EPSG:4326", transform=TRANSFORM) as dst:
        dst.write(array, 1)

def phase1_reproject(inputs, work):
    """WGS84 -> EPSG:25829 for every raster input (mirrors src/crs_handler.py)."""
    for key in ["vh", "vv", "nir", "red"]:
        raw_path = outpath(f"raw_{key}.tif")
        write_tif(raw_path, inputs[key])
        with rasterio.open(raw_path) as src:
            transform, width, height = calculate_default_transform(
                src.crs, "EPSG:25829", src.width, src.height, *src.bounds)
            kwargs = src.meta.copy()
            kwargs.update({"crs": "EPSG:25829", "transform": transform,
                           "width": width, "height": height})
            with rasterio.open(work[key], "w", **kwargs) as dst:
                reproject(source=rasterio.band(src, 1), destination=rasterio.band(dst, 1),
                          src_transform=src.transform, src_crs=src.crs,
                          dst_transform=transform, dst_crs="EPSG:25829",
                          resampling=Resampling.bilinear)

def phase2_forest_mask(work):
    """SAR cloud-proof forest mask (mirrors src/sar_optical_fusion.py)."""
    with rasterio.open(work["vh"]) as vhr, rasterio.open(work["vv"]) as vvr:
        vh = vhr.read(1).astype("float32")
        vv = vvr.read(1).astype("float32")
    with np.errstate(divide="ignore"):
        vh_db = 10 * np.log10(np.where(vh > 0, vh, 1e-10))
    ratio = np.divide(vh, vv, out=np.zeros_like(vh), where=vv != 0)
    mask = ((vh_db > -15) & (ratio > 0.15)).astype("uint8")
    write_tif(work["mask"], mask)
    return mask

def phase3_nfi(inputs):
    """DBH filter + hypsometer heights + reprojection (mirrors src/nfi_integration.py)."""
    df = inputs["nfi"]
    valid = df[df["DBH"] >= 10.0].copy()
    valid["Height"] = valid["Distance"] * np.tan(np.radians(valid["Angle"])) + valid["ObsHeight"]
    gdf = gpd.GeoDataFrame(valid, geometry=gpd.points_from_xy(valid["x"], valid["y"]),
                           crs="EPSG:4326")
    return gdf.to_crs("EPSG:25829")

def phase4_species(inputs, work):
    """Random Forest species classification (production path: U-Net in src/ml_models.py)."""
    X = inputs["features"].reshape(6, -1).T
    y = inputs["species_truth"].ravel()
    model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    model.fit(X, y)
    pred = model.predict(X).reshape(H, W).astype("uint8")
    write_tif(work["species"], pred)
    return model, pred

def phase5_carbon(species_map, work):
    """Tier-2 biomass + +/-95% CI (mirrors src/carbon_accounting.py)."""
    biomass = np.zeros_like(species_map, dtype="float32")
    unc = np.zeros_like(species_map, dtype="float32")
    for sp, peak in BIOMASS_PEAK.items():
        m = species_map == sp
        biomass[m] = peak * 0.8
        unc[m] = biomass[m] * UNCERTAINTY_FACTOR[sp] * 1.96
    with rasterio.open(work["carbon"], "w", driver="GTiff", height=H, width=W, count=2,
                       dtype="float32", crs="EPSG:4326", transform=TRANSFORM) as dst:
        dst.write(biomass, 1)
        dst.write(unc, 2)
        dst.set_band_description(1, "biomass_t_per_ha")
        dst.set_band_description(2, "ci95_t_per_ha")
    return biomass, unc

def embed_provenance(src_path, dst_path, input_hash):
    """Copy a raster with provenance tags (mirrors src/metadata_embedder.py)."""
    try:
        git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"],
                                          stderr=subprocess.DEVNULL).decode().strip()
    except Exception:
        git_sha = "colab-local-run"
    with rasterio.open(src_path) as src:
        meta = src.meta.copy()
        data = src.read()
    meta.update(PROVENANCE_COMMIT_SHA=git_sha, INPUT_HASH=input_hash, PIPELINE_VERSION="1.1.0")
    with rasterio.open(dst_path, "w", **meta) as dst:
        dst.write(data)
    return git_sha

print("Phases defined: 1 CRS | 2 SAR mask | 3 NFI | 4 species | 5 carbon | quarantine | provenance")

In [ ]:
# ============================================================
# The orchestrator: run every phase FOR REAL, end to end
# ============================================================
def run_pipeline(seed=42, tag="synthetic"):
    t0 = time.time()
    inputs = generate_all_inputs(seed)
    work = {"vh": outpath("p1_vh_utm.tif"), "vv": outpath("p1_vv_utm.tif"),
            "nir": outpath("p1_nir_utm.tif"), "red": outpath("p1_red_utm.tif"),
            "mask": outpath("p2_forest_mask.tif"), "species": outpath("p4_species_map.tif"),
            "carbon": outpath("p5_biomass_uncertainty.tif")}
    report_rows = []

    t = time.time()
    phase1_reproject(inputs, work)
    report_rows.append({"Phase": "1. CRS reproject -> EPSG:25829", "Status": "OK",
                        "Seconds": round(time.time() - t, 2)})
    print("✅ Phase 1: all rasters reprojected to EPSG:25829")

    t = time.time()
    mask = phase2_forest_mask(work)
    report_rows.append({"Phase": "2. SAR forest mask", "Status": "OK",
                        "Seconds": round(time.time() - t, 2),
                        "Detail": f"{int(mask.sum())} forest pixels"})
    print(f"✅ Phase 2: SAR forest mask ({int(mask.sum())} forest pixels, cloud-proof)")

    t = time.time()
    gdf = phase3_nfi(inputs)
    report_rows.append({"Phase": "3. NFI validation + hypsometer heights", "Status": "OK",
                        "Seconds": round(time.time() - t, 2),
                        "Detail": f"{len(gdf)}/{len(inputs['nfi'])} plots pass DBH >= 10 cm"})
    print(f"✅ Phase 3: NFI validated ({len(gdf)}/{len(inputs['nfi'])} plots kept, heights computed)")

    t = time.time()
    model, species_map = phase4_species(inputs, work)
    report_rows.append({"Phase": "4. Species classification (Random Forest)", "Status": "OK",
                        "Seconds": round(time.time() - t, 2)})
    print("✅ Phase 4: species map classified (Random Forest, 100 trees)")

    t = time.time()
    biomass, unc = phase5_carbon(species_map, work)
    report_rows.append({"Phase": "5. Tier-2 carbon + uncertainty", "Status": "OK",
                        "Seconds": round(time.time() - t, 2),
                        "Detail": f"total ~{float(biomass.mean()) * H * W * PIXEL_AREA_HA:.0f} t biomass"})
    print("✅ Phase 5: biomass + uncertainty written (2-band GeoTIFF)")

    # ---- Fail-loud quarantine demo: a corrupted tile never fails silently ----
    corrupt = inputs["vh"].copy()
    corrupt[40:80, 40:80] = np.nan
    corrupt_path = outpath("corrupted_tile.tif")
    write_tif(corrupt_path, corrupt)
    try:
        if not np.isfinite(corrupt).all():
            raise ValueError("non-finite (NaN/Inf) backscatter values in VH tile")
    except Exception as e:
        qpath = log_quarantine("tile_40_80", "NAN_BACKSCATTER", str(e),
                               input_hash=sha256_of(corrupt_path)[:16])
        make_download_button(qpath, label="⬇️ Download quarantine log (.json)",
                             register_as=f"Quarantine log ({tag})")

    # ---- Provenance embedding on the final products ----
    input_hash = sha256_of(outpath("raw_vh.tif"))[:16]
    git_sha = embed_provenance(work["mask"], outpath("final_forest_mask_provenanced.tif"),
                               input_hash)
    embed_provenance(work["carbon"], outpath("final_carbon_provenanced.tif"), input_hash)
    print(f"✅ Provenance: SHA={git_sha[:8]}, INPUT_HASH={input_hash} embedded into final GeoTIFFs")

    report = pd.DataFrame(report_rows)
    report_path = outpath("provenance_report.xlsx")
    with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
        report.to_excel(writer, sheet_name="Phases", index=False)
        pd.DataFrame({"Key": ["PROVENANCE_COMMIT_SHA", "INPUT_HASH", "PIPELINE_VERSION", "RUN_SEED"],
                      "Value": [git_sha, input_hash, "1.1.0", seed]}).to_excel(
            writer, sheet_name="Provenance", index=False)

    fig, axes = plt.subplots(2, 2, figsize=(11, 8.5))
    axes[0, 0].imshow(mask, cmap="Greens")
    axes[0, 0].set_title("P2: SAR forest mask (cloud-proof)")
    axes[0, 1].imshow(species_map, cmap="terrain", vmin=1, vmax=3)
    axes[0, 1].set_title("P4: species map (1/2/3)")
    im = axes[1, 0].imshow(biomass, cmap="YlOrRd", vmin=0, vmax=400)
    axes[1, 0].set_title("P5: biomass (t/ha)")
    plt.colorbar(im, ax=axes[1, 0], fraction=0.046)
    im2 = axes[1, 1].imshow(unc, cmap="Purples", vmin=0, vmax=120)
    axes[1, 1].set_title("P5: uncertainty (+/-95% CI, t/ha)")
    plt.colorbar(im2, ax=axes[1, 1], fraction=0.046)
    for ax in axes.flat:
        ax.set_axis_off()
    plt.tight_layout()
    dash_path = outpath("step9_dashboard.png")
    fig.savefig(dash_path, dpi=120, bbox_inches="tight")
    plt.show()

    for label, p in [("dashboard chart", dash_path), ("provenance report", report_path),
                     ("final forest mask (provenanced)", outpath("final_forest_mask_provenanced.tif")),
                     ("final carbon raster (provenanced)", outpath("final_carbon_provenanced.tif"))]:
        make_download_button(p, label=f"⬇️ Download {label}", register_as=f"{label} ({tag})")

    print(f"✅ Orchestrator finished in {time.time() - t0:.1f}s - every phase ran for real.")
    return mask, species_map, biomass, unc

mask, species_map, biomass, unc = run_pipeline(42)

**Interpretation:** this dashboard is the pipeline's actual product. Top-left: the SAR forest
mask survived even though the "scene" has a peat/water block — physics thresholds did that. Top-right:
the species partition that drives the biomass matrix. Bottom row: the LULUCF-ready numbers —
biomass and its explicit ±95% CI. Note the quarantine line in the log above: the corrupted tile
was **caught, hashed, logged, and skipped** — the run never stopped and never lied. In production
the same JSON lands in `quarantine/`, and the provenance sheet is what lets an auditor replay
this exact run years later.

In [ ]:
# ============================================================
# Regenerate control - one seed re-runs the ENTIRE pipeline
# ============================================================
def rerun_pipeline(seed=None):
    """Fresh seed -> fully regenerated inputs + every phase re-executed."""
    seed = seed or np.random.randint(0, 10_000)
    print(f"[seed={seed}] full pipeline re-run starting...")
    run_pipeline(seed, tag=f"seed-{seed}")

make_regenerate_button(rerun_pipeline, label="🔄 Regenerate inputs & re-run entire pipeline")
make_download_all_button()